<a href="https://colab.research.google.com/github/Regge12/CS-Project-ORCA/blob/main/Text_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Classification (Reviews)
Classifying reviews to be negative or positive

In [13]:
import re
import string

# Stop words: common words with no sentiment value, removed during preprocessing
STOPWORDS = {
    "the", "a", "an", "is", "it", "this", "that", "was", "and",
    "of", "to", "in", "i", "my", "for", "with", "on", "at", "be",
    "are", "have", "had", "but", "or", "as", "so", "we", "he",
    "she", "they", "its", "by", "from", "about", "been"
}

# Negation words: when detected, the sentiment of the NEXT matched word is flipped
NEGATION_WORDS = {"not", "never", "no", "hardly", "barely"}

# Positive patterns: regex patterns matched against individual tokens
# Using regex allows partial matching, e.g. "enjoy" matches "enjoyable", "enjoyed"
POSITIVE_PATTERNS = [
    r"enjoy\w*",       # enjoy, enjoyed, enjoyable
    r"brilliant\w*",
    r"beauti\w*",      # beautiful, beautifully
    r"great\w*",
    r"outstand\w*",    # outstanding
    r"excel\w*",       # excellent, excels
    r"perfect\w*",
    r"love\w*",        # love, loved, lovely
    r"amaz\w*",        # amazing, amazed
    r"wonderf\w*",     # wonderful
    r"masterpi\w*",    # masterpiece
    r"impress\w*",     # impressive, impressed
    r"captivat\w*",    # captivating
    r"heartwarming",
    r"recommend\w*",
    r"inspir\w*",      # inspiring, inspired
]

# Negative patterns
NEGATIVE_PATTERNS = [
    r"terr\w*",        # terrible, terribly
    r"awful\w*",
    r"bore\w*",        # boring, bored
    r"disappoint\w*",  # disappointing, disappointed
    r"worst\w*",
    r"bad\w*",
    r"dull\w*",
    r"poor\w*",
    r"painf\w*",       # painful, painfully
    r"horr\w*",        # horrible, horrific
    r"waste\w*",
    r"ridicul\w*",     # ridiculous
    r"unbearabl\w*",
    r"confus\w*",      # confusing, confused
    r"predictabl\w*",
]

In [12]:
def prepare_text(text):
    """
    Preprocesses a review for classification.
    Steps: remove HTML tags → lowercase → remove punctuation → tokenise → remove stop words
    """
    # Step 1: Remove any HTML tags (e.g. <br />) from the raw review text
    text = re.sub(r"<[^>]+>", " ", text)

    # Step 2: Convert to lowercase
    text = text.lower()

    # Step 3: Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Step 4: Tokenise
    tokens = text.split()

    # Step 5: Remove stop words
    tokens = [t for t in tokens if t not in STOPWORDS]

    return tokens

In [4]:
def classify_review(text):
    """
    Classifies a film review as 'positive' or 'negative'.
    Uses keyword pattern matching with negation handling.
    Returns: predicted label (str) and a reasoning trace (dict)
    """
    tokens = prepare_text(text)
    pos_score = 0
    neg_score = 0
    negation_active = False
    reasoning = []  # stores a trace of every decision for explainability

    for i, token in enumerate(tokens):

        # Check for negation words first
        if token in NEGATION_WORDS:
            negation_active = True
            reasoning.append(f"  [Token '{token}'] → Negation detected, flag set to True")
            continue  # move to next token without scoring

        matched = False

        # Check against positive patterns
        for pattern in POSITIVE_PATTERNS:
            if re.fullmatch(pattern, token):
                if negation_active:
                    neg_score += 1
                    reasoning.append(f"  [Token '{token}'] → Positive pattern match, BUT negation active → neg_score +1")
                else:
                    pos_score += 1
                    reasoning.append(f"  [Token '{token}'] → Positive pattern match → pos_score +1")
                negation_active = False
                matched = True
                break

        if matched:
            continue

        # Check against negative patterns
        for pattern in NEGATIVE_PATTERNS:
            if re.fullmatch(pattern, token):
                if negation_active:
                    pos_score += 1
                    reasoning.append(f"  [Token '{token}'] → Negative pattern match, BUT negation active → pos_score +1")
                else:
                    neg_score += 1
                    reasoning.append(f"  [Token '{token}'] → Negative pattern match → neg_score +1")
                negation_active = False
                matched = True
                break

        # If no match and not a negation word, reset negation flag
        if not matched:
            negation_active = False

    # Decision rule
    if pos_score > neg_score:
        label = "positive"
    else:
        label = "negative"  # default for ties and neg dominance

    return label, pos_score, neg_score, reasoning



In [5]:
def evaluate(reviews, true_labels):
    """
    Evaluates classifier performance across a dataset.
    Returns accuracy, precision, recall, and F1 score.
    """
    correct = 0
    true_positive = 0
    false_positive = 0
    false_negative = 0

    for i in range(len(reviews)):
        prediction, _, _, _ = classify_review(reviews[i])
        if prediction == true_labels[i]:
            correct += 1
        if prediction == "positive" and true_labels[i] == "positive":
            true_positive += 1
        if prediction == "positive" and true_labels[i] == "negative":
            false_positive += 1
        if prediction == "negative" and true_labels[i] == "positive":
            false_negative += 1

    accuracy  = correct / len(reviews)

    # Guard against division by zero
    precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
    recall    = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return accuracy, precision, recall, f1

In [33]:
import json

# Load reviews from JSON file
# Each entry is in the format {"review": "...", "label": "..."}
with open("reviews.json", "r") as f:
    data = json.load(f)

# Split into separate lists using dictionary keys
reviews     = [entry["review"] for entry in data]
true_labels = [entry["label"]  for entry in data]

for i in range(len(reviews)):
  label, pos_score, neg_score, reasoning = classify_review(reviews[i])
  if pos_score == neg_score:
    # Mix sentiment neutral
    # print("\n" + reviews[i]);
    pass

print(f"Loaded {len(reviews)} reviews successfully")
print(f"  Positive: {true_labels.count('positive')}")
print(f"  Negative: {true_labels.count('negative')}")

# Example of test cases

# Clearly possitive
for i in range(len(reviews)):
  if re.match("Probably my all-time favorite movie, a story of selflessness,", reviews[i]):
    print("\n"+reviews[i])

#Clearly negative
for i in range(len(reviews)):
  if re.match("Of all the films I have seen, this one, The Rage, has got", reviews[i]):
    print("\n"+reviews[i])

#Mixed sentiment
for i in range(len(reviews)):
  if re.match("This movie has exactly the same number of things I liked and disliked", reviews[i]):
    print("\n"+reviews[i])

#Sarcasm
for i in range(len(reviews)):
  if re.match("Oh yes, this was obviously the greatest movie", reviews[i]):
    print("\n"+reviews[i])



Loaded 65 reviews successfully
  Positive: 31
  Negative: 34

Probably my all-time favorite movie, a story of selflessness, sacrifice and dedication to a noble cause, but it's not preachy or boring. It just never gets old, despite my having seen it some 15 or more times in the last 25 years. Paul Lukas' performance brings tears to my eyes, and Bette Davis, in one of her very few truly sympathetic roles, is a delight. The kids are, as grandma says, more like "dressed-up midgets" than children, but that only makes them more fun to watch. And the mother's slow awakening to what's happening in the world and under her own roof is believable and startling. If I had a dozen thumbs, they'd all be "up" for this movie.

Of all the films I have seen, this one, The Rage, has got to be one of the worst yet. The direction, LOGIC, continuity, changes in plot-script and dialog made me cry out in pain. "How could ANYONE come up with something so crappy"? Gary Busey is know for his "B" movies, but this 

In [16]:
accuracy, precision, recall, f1 = evaluate(reviews, true_labels)

print("── Evaluation Results ──────────────────────")
print(f"Accuracy  : {accuracy:.2%}")
print(f"Precision : {precision:.2%}")
print(f"Recall    : {recall:.2%}")
print(f"F1 Score  : {f1:.2%}")

── Evaluation Results ──────────────────────
Accuracy  : 55.00%
Precision : 50.00%
Recall    : 44.44%
F1 Score  : 47.06%
